In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = "ecommerce"

### Brands

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn("brand_name", trim(col("brand_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("brand_code", regexp_replace(col("brand_code"), r'[^A-Za-z0-9]', ""))
df_silver.show(10)

In [0]:
df_silver.select(col("category_code")).distinct().show()

In [0]:
anomalies = {
    "TOYS":"TOY", 
    "BOOKS":"BKS", 
    "GROCERY":"GRCY"
}
df_silver = df_silver.replace(anomalies, subset=["category_code"])
df_silver.select(col("category_code")).distinct().show()

In [0]:
df_silver.write.format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{catalog_name}.silver.slv_brands")

### Category

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_bronze.select(col("category_code")).distinct().show()

In [0]:
df_silver = df_bronze.withColumn("category_code", upper(col("category_code")))
df_silver.show(20)

In [0]:
df_silver.groupBy("category_code").agg(count("category_name").alias("category_name_count")).filter(col("category_name_count") > 1).show()

In [0]:
df_silver = df_silver.dropDuplicates(["category_code"])
df_silver.show()

In [0]:
df_silver.write.format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{catalog_name}.silver.slv_category")

---
### Products

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
df_bronze.show(10)

In [0]:
df_bronze.groupBy('product_id').count().filter(col("count") > 1).show()

In [0]:
df_silver = df_bronze.withColumn("category_code", trim(upper(col("category_code"))))\
            .withColumn("brand_code", trim(upper(col("brand_code"))))

df_silver.show(10)

In [0]:
df_silver.select("color").distinct().show()

In [0]:
df_silver.select("size").distinct().show()

In [0]:
df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn("weight_grams", regexp_replace(col("weight_grams"), "g", "").cast(IntegerType()))
df_silver.show(10)

In [0]:
df_silver.printSchema()

In [0]:
df_silver = df_silver.withColumn("length_cm", regexp_replace(col("length_cm"), ",", ".").cast(FloatType()))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("material", when(col("material") == "Coton", "Cotton")\
                                .when(col("material") == "Alumium", "Aluminum")\
                                .when(col("material") == "Ruber", "Rubber")\
                                .otherwise(col("material")))

df_silver.show(10)

In [0]:
df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn("rating_count", when(col("rating_count").isNotNull(), abs(col("rating_count")))\
                                    .otherwise(lit(0)))

df_silver.show(10)

In [0]:
df_silver.write.format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{catalog_name}.silver.slv_products")

### Customers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_bronze.show(10)

In [0]:
df_bronze.select("*").filter(col("customer_id").isNull()).count()

In [0]:
df_bronze = df_bronze.dropDuplicates(["customer_id"])

In [0]:
df_bronze.count()

In [0]:
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_silver = df_bronze.withColumn("phone", when(col("phone").isNotNull(), regexp_replace(col("phone"), r"\.0", ""))\
                                .otherwise(col("phone")))

df_silver.show(10)

In [0]:
df_silver = df_silver.fillna("Not Available", "phone")
df_silver.show(10)

In [0]:
df_silver.select("country_code", "country").distinct().show()

In [0]:
anomalies = {
    "GB":"UK",
    "AE":"UAE"
}
df_silver = df_silver.replace(anomalies, subset="country_code")
df_silver.show(10)

In [0]:
df_silver.select("country_code", "country").distinct().show()

In [0]:
df_silver.select("state").distinct().show(30)

In [0]:
df_silver.write.format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{catalog_name}.silver.slv_customers")

### Date

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_calendar")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_silver = df_bronze.withColumn("day_name", upper(col("day_name")))
df_silver.show(10)

In [0]:
df_silver.select("day_name").distinct().show()

In [0]:
df_silver.select("year").filter((col("year") > 2026) | (col("year") < 1900)).show()

In [0]:
df_silver.select("quarter").filter((col("quarter") > 4) | (col("quarter") < 1)).show()

In [0]:
df_silver.select("week_of_year").filter((col("week_of_year") >= 0) | (col("week_of_year") < -52)).show()

In [0]:
df_silver = df_silver.withColumn("date", to_date(col("date"), "dd-MM-yyyy"))
df_silver.show(10)
df_silver.printSchema()

In [0]:
duplicates = df_silver.groupBy("date").count().filter(col("count") > 1)

print(f"Total duplicated rows: {duplicates.count()}")
duplicates.show()

In [0]:
df_silver = df_silver.dropDuplicates(["date"])

In [0]:
df_silver.groupBy("date").count().filter(col("count") > 1).show()

In [0]:
df_bronze.show(10)

In [0]:
df_silver = df_silver.withColumn("day_name", initcap(col("day_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("week_of_year", abs(col("week_of_year")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("quarter", concat_ws("", lit("Q"), col("quarter"), lit("-"), col("year")))\
            .withColumn("week_of_year", concat_ws("", lit("Week"), col("week_of_year"), lit("-"), col("year")))

df_silver.show(10)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")
df_silver.show(3)

In [0]:
df_silver.write.format("delta")\
                .mode("overwrite")\
                .option("mergeSchema", True)\
                .saveAsTable(f"{catalog_name}.silver.slv_calendar")